In [8]:
import json
import onnx
import torch
import numpy as np
import pandas as pd
from torch import nn

In [9]:
class ExampleModel(nn.Module):

    def __init__(self):
        super(ExampleModel, self).__init__()
        self.x_dim = 3
        self.u_dim = 2
        self.y_dim = 1

        # set random seed for reproducibility
        torch.manual_seed(42)

        self.dynamic = nn.Sequential(
            nn.Linear(self.x_dim + self.u_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
        self.output = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, u, x_0):
        x_1 = self.dynamic(torch.cat((x_0, u), dim=0))
        y_1 = self.output(x_1)
        return y_1, x_1

In [10]:
# Create the model
model = ExampleModel()


# Create three tensors
u = torch.tensor([0.5, -0.2], dtype=torch.float32)  # Control input
x_0 = torch.tensor([1.0, 0.0, -1.0], dtype=torch.float32)  # Initial state

# Run the model
y_1, x_1 = model(u, x_0)

print("Output y:", y_1[0].item())

Output y: 0.2299169898033142


In [11]:
model_name = "example5"
# Set the model to evaluation mode
model.eval()

# Save the model in ONNX format
torch.onnx.export(
    model,
    (u, x_0),
    f"{model_name}.onnx",
    verbose=False,
    input_names=["u", "x"],
    output_names=["ynext", "xnext"],
)

# Load the model
onnx_model = onnx.load(f"{model_name}.onnx")

# Check the model
onnx.checker.check_model(onnx_model)

# Add description to the model
onnx_model.graph.doc_string = "Example to test FMU with local variables."

# Add metadata to the model
onnx_model.producer_name = "ExampleModel"
onnx_model.producer_version = "0.0.1"
onnx_model.domain = "example"
onnx_model.model_version = 1

# Save the model
onnx.save(onnx_model, f"{model_name}.onnx")


## Generating model description

Create and save the model description to be provided to ONNX2FMU.

In [12]:
model_description = {
    "name": "example5",
    "description": "Example to test FMU with local variables.",
    "FMIVersion": "2.0",
    "inputs": [
        {
            "name": "u",
            "description": "A vector of control variables at time t, size 2."
        },
    ],
    "outputs": [
        {
            "name": "ynext",
            "description": "The output variable at time t+1, size 1."
        }
    ],
    "locals": [
        {
            "nameIn": "x",
            "nameOut": "xnext",
            "description": "The local state variable, size 3.",
            "start": [
                0.0,
                0.0,
                0.0
            ]
        },
    ]
}

# Save model description
with open(f"{model_name}Description.json", "w", encoding="utf-8") as f:
    json.dump(model_description, f, indent=4)

## Generating input file and output for testing

In [13]:
time_steps = 100
u_dim = model.u_dim
x_dim = model.x_dim

# create and save input history
inputs = np.ones((time_steps, u_dim)) * np.arange(time_steps)[:, None]
columns = [f"u_0_{i}" for i in range(u_dim)]
index = pd.Index(np.arange(time_steps), name='time')
pd.DataFrame(data=inputs, columns=columns, index=index).to_csv("input.csv")

# buffers for outputs and states
results_y = torch.empty((time_steps, model.y_dim), dtype=torch.float32)
results_x = torch.empty((time_steps, x_dim), dtype=torch.float32)

# initial state
x_prev = torch.zeros(x_dim, dtype=torch.float32) # as in model description

for i in range(time_steps):
    u_row = torch.tensor(inputs[i], dtype=torch.float32)
    ynext, xnext = model(u_row, x_prev)    # new model: (u, x) -> (ynext, xnext)
    results_y[i] = ynext.detach().reshape(-1)
    results_x[i] = xnext.detach().reshape(-1)
    x_prev = xnext.detach()

# output into one CSV
output = pd.DataFrame(
    data=results_y.numpy(),
    columns=[f"ynext_{i}" for i in range(model.y_dim)],
    index=index
)
output.to_csv("output.csv")
# states into separate CSV
states = pd.DataFrame(
    data=results_x.numpy(),
    columns=[f"x{i}" for i in range(x_dim)],
    index=index
)
states.to_csv("states.csv")
